# VFund quickstart

A 5-minute tour: run an honest backtest on synthetic data (no network, no API keys), then a cross-sectional long/short book. Install first: `pip install -e ".[dev]"` from the repo root.

## 1. A single-asset backtest

The engine decides on today's close and fills at the *next* bar (no look-ahead), and charges costs. Here a moving-average crossover on synthetic data.

In [ ]:
from vfund.data.synthetic import generate_gbm_bars
from vfund.backtest import Backtester
from vfund.strategy import MACrossover

data = generate_gbm_bars(2000, interval="1h", seed=1)
result = Backtester(data, MACrossover(fast=20, slow=50), interval="1h").run()
print(result.summary())

## 2. A cross-sectional long/short book

Rank many coins against each other. This synthetic panel has a real reversal effect baked in, so a correct engine should profit on it.

In [ ]:
from vfund.data.synthetic import generate_gbm_panel
from vfund.backtest.cross_sectional import CrossSectionalBacktester
from vfund.strategy import CrossSectionalReversal

panel = generate_gbm_panel(20, 3000, reversion=0.2, seed=1)
res = CrossSectionalBacktester(panel, CrossSectionalReversal(1), rebalance_every=1, cost_bps=0).run()
m = res.metrics()
print(f"Sharpe {m['sharpe']:.2f} | total return {m['total_return']*100:.0f}% | max DD {m['max_drawdown']*100:.0f}%")

## 3. The honest test: out-of-sample

A good backtest means little; surviving *unseen* data is what matters. See `examples/oos_gauntlet.py` for selecting in-sample and judging out-of-sample, and `docs/CASE_STUDY.md` for the full research story.